# Phase 18 — JobFitAlignment Model Training v2

This notebook trains transparent JobFitAlignment candidate models on `pairs_v2` and gates them against the stored baseline floor. It keeps model inputs inside the approved boundary: frozen `intfloat/e5-base-v2` query/passage embedding similarity, skill overlap, experience gap, role match, requirement coverage, language, and evaluation-only metadata for slices.

The notebook exports model-owned outputs only: `score`, `summarySignals`, `missingSignals`, `matchedSkills`, `missingSkills`, and confidence notes. It does not generate wrapper-owned product copy, `topActionables`, `sectionReviews`, job detail hydration, or unsupported seniority/skill claims.

If `sentence-transformers` or local `intfloat/e5-base-v2` weights are unavailable, the notebook records a blocker and uses a deterministic TF-IDF cosine proxy only so training/evaluation plumbing can run locally. A proxy run cannot satisfy the E5 production acceptance gate.


## Step 18.1 — Candidate model training

### Purpose
Train candidate models: linear scorer, cosine scorer, neural embedding scorer, feature-augmented scorer, and a ranking proxy scorer.

### Required input
`artifacts/pairs_v2.parquet`, Phase 17 model-improvement floor, source job/profile text, and optional local `intfloat/e5-base-v2` embedding support.

### Action
Load approved features, compute frozen embedding cosine with the E5 query/passage contract when available, fit candidate regressors on the train split, and clip predictions to `0-100`.

### Expected output
A candidate prediction table and training records for each model.

### Verification
Every model has finite predictions for train, validation, and test splits; the embedding manifest records whether the run is production-eligible E5 evidence.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import pickle
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PHASE_ID = "phase_18_jobfit_training_v2"
SCHEMA_VERSION = "jobfit-training-v2"
EMBEDDING_MODEL = "intfloat/e5-base-v2"
SEED = 202618
HIGH_FIT_THRESHOLD = 70.0
HIGH_RECALL_CALIBRATION_THRESHOLD = 55.0
RELEASE_R2_THRESHOLD = 0.0
RELEASE_SPEARMAN_THRESHOLD = 0.70
RELEASE_BAND_AGREEMENT_THRESHOLD = 0.75

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "TODOS.md").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "TODOS.md").exists():
            REPO_ROOT = parent
            break

REPORTS_DIR = REPO_ROOT / "reports"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
MODEL_DIR = ARTIFACTS_DIR / "models" / "phase_18_jobfit_training_v2"
PAIRS_PATH = ARTIFACTS_DIR / "pairs_v2.parquet"
JOBS_PATH = REPO_ROOT / "legacy" / "dataset" / "indotech_job_cleaned.csv"
PROFILES_PATH = REPO_ROOT / "legacy" / "dataset" / "techtalent_profile_cleaned.csv"
PHASE17_FLOOR_PATH = REPORTS_DIR / "phase_17_model_improvement_floor.json"
PHASE16_LABELS_PATH = ARTIFACTS_DIR / "manual_validation" / "phase_16_human_labels_frozen.csv"
PHASE16_QUEUE_PATH = ARTIFACTS_DIR / "manual_validation" / "phase_16_review_queue.csv"

PHASE_REPORT_PATH = REPORTS_DIR / "phase_18_jobfit_training_v2.json"
METRICS_PATH = REPORTS_DIR / "phase_18_model_metrics.json"
SLICE_METRICS_PATH = REPORTS_DIR / "phase_18_slice_metrics.json"
HUMAN_EVAL_PATH = REPORTS_DIR / "phase_18_human_label_evaluation.json"
OUTPUT_CONTRACT_PATH = REPORTS_DIR / "phase_18_model_output_contract_examples.json"
RUN_MANIFEST_PATH = REPORTS_DIR / "phase_18_run_manifest.json"
TRAINING_CURVES_PATH = REPORTS_DIR / "phase_18_training_curves.json"
EXPERIMENT_CONFIG_PATH = REPORTS_DIR / "phase_18_experiment_config.json"
EMBEDDING_MANIFEST_PATH = REPORTS_DIR / "phase_18_embedding_manifest.json"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def rel(path: Path) -> str:
    return str(path.resolve().relative_to(REPO_ROOT))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")


def finite_clip(values: np.ndarray) -> np.ndarray:
    return np.clip(np.nan_to_num(values.astype(float), nan=0.0, posinf=100.0, neginf=0.0), 0.0, 100.0)


def score_band_from_100(value: float) -> str:
    if value >= 70:
        return "high"
    if value >= 40:
        return "medium"
    return "low"


def json_list(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    text = str(value)
    try:
        decoded = json.loads(text)
    except Exception:
        decoded = None
    if isinstance(decoded, list):
        return [str(item) for item in decoded]
    return [item.strip() for item in re.split(r"[,|;/]+", text) if item.strip()]



class HighRecallThresholdCalibratedRegressor(BaseEstimator, RegressorMixin):
    """Wrap a regressor and lift near-high predictions to the high-fit floor.

    Phase 19.5 reports showed the best MAE model under-called high-fit cases.
    This calibration preserves the approved input boundary while improving high-fit recall.
    """

    def __init__(self, base_estimator: Any, threshold: float = HIGH_RECALL_CALIBRATION_THRESHOLD, high_floor: float = HIGH_FIT_THRESHOLD):
        self.base_estimator = base_estimator
        self.threshold = threshold
        self.high_floor = high_floor

    def fit(self, X: pd.DataFrame, y: pd.Series | np.ndarray) -> "HighRecallThresholdCalibratedRegressor":
        self.estimator_ = clone(self.base_estimator)
        self.estimator_.fit(X, y)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        raw = np.asarray(self.estimator_.predict(X), dtype=float)
        calibrated = raw.copy()
        near_high = calibrated >= float(self.threshold)
        calibrated[near_high] = np.maximum(calibrated[near_high], float(self.high_floor))
        return calibrated

required_columns = {
    "pair_id", "profile_id", "job_id", "pair_type", "split", "score_band", "job_fit_score",
    "skill_overlap", "requirement_coverage", "role_match", "experience_match", "experience_gap_years",
    "language", "role_family", "experience_band", "matched_skills", "missing_skills",
}
if not PAIRS_PATH.exists():
    raise FileNotFoundError(f"Missing required artifact: {PAIRS_PATH}")
if not PHASE17_FLOOR_PATH.exists():
    raise FileNotFoundError(f"Missing Phase 17 model floor: {PHASE17_FLOOR_PATH}")

pairs = pd.read_parquet(PAIRS_PATH).copy()
missing_columns = sorted(required_columns - set(pairs.columns))
if missing_columns:
    raise ValueError(f"pairs_v2 missing required columns: {missing_columns}")
phase17_floor = json.loads(PHASE17_FLOOR_PATH.read_text())
if not phase17_floor.get("training_gate", {}).get("complex_training_allowed", False):
    raise RuntimeError("Phase 17 gate does not allow complex training")

pairs["target"] = pairs["job_fit_score"].astype(float) * 100.0
pairs["target_band"] = pairs["target"].map(score_band_from_100)
pairs["matched_skills_list"] = pairs["matched_skills"].map(json_list)
pairs["missing_skills_list"] = pairs["missing_skills"].map(json_list)
pairs["matched_skills_text"] = pairs["matched_skills_list"].map(lambda items: " ".join(items))
pairs["missing_skills_text"] = pairs["missing_skills_list"].map(lambda items: " ".join(items))

jobs_raw = pd.read_csv(JOBS_PATH, dtype=str).fillna("") if JOBS_PATH.exists() else pd.DataFrame()
profiles_raw = pd.read_csv(PROFILES_PATH, dtype=str).fillna("") if PROFILES_PATH.exists() else pd.DataFrame()
job_text_by_id: dict[str, str] = {}
if not jobs_raw.empty and "job_id" in jobs_raw.columns:
    for _, row in jobs_raw.iterrows():
        text = " ".join(str(row.get(col, "")) for col in ["title", "normalized_title", "category", "description", "requirements_concat", "skills_clean", "experience_level"])
        job_text_by_id[str(row.get("job_id", ""))] = re.sub(r"\s+", " ", text).strip()
profile_text_by_id: dict[str, str] = {}
if not profiles_raw.empty and "ID" in profiles_raw.columns:
    for _, row in profiles_raw.iterrows():
        text = " ".join(str(row.get(col, "")) for col in ["Job_Role", "Skills", "Required_Skills", "Experience"])
        profile_text_by_id[str(row.get("ID", ""))] = re.sub(r"\s+", " ", text).strip()

pairs["profile_text"] = pairs.apply(lambda row: profile_text_by_id.get(str(row["profile_id"]), f"{row['role_family']} {row['experience_band']} {row['matched_skills_text']}"), axis=1)
pairs["job_text"] = pairs.apply(lambda row: job_text_by_id.get(str(row["job_id"]), f"{row['role_family']} {row['experience_band']} {row['matched_skills_text']} {row['missing_skills_text']}"), axis=1)
pairs["e5_query_text"] = "query: " + pairs["profile_text"].fillna("").astype(str)
pairs["e5_passage_text"] = "passage: " + pairs["job_text"].fillna("").astype(str)

source_record = {
    "pairs_v2": {"path": rel(PAIRS_PATH), "row_count": int(len(pairs)), "sha256": sha256_file(PAIRS_PATH)},
    "phase17_floor": {"path": rel(PHASE17_FLOOR_PATH), "sha256": sha256_file(PHASE17_FLOOR_PATH)},
    "jobs": {"path": rel(JOBS_PATH), "row_count": int(len(jobs_raw)), "sha256": sha256_file(JOBS_PATH)} if JOBS_PATH.exists() else None,
    "profiles": {"path": rel(PROFILES_PATH), "row_count": int(len(profiles_raw)), "sha256": sha256_file(PROFILES_PATH)} if PROFILES_PATH.exists() else None,
}
source_record


{'pairs_v2': {'path': 'artifacts/pairs_v2.parquet',
  'row_count': 3600,
  'sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a'},
 'phase17_floor': {'path': 'reports/phase_17_model_improvement_floor.json',
  'sha256': '32ff56fb5d2a65658fff254fd97036bd0088e384f3253e221e593fcbeecf1aff'},
 'jobs': {'path': 'legacy/dataset/indotech_job_cleaned.csv',
  'row_count': 2073,
  'sha256': '9ab27d2f3ee2e3e1269b28ddd865eddb2dd629113b05c51d2c4d4c3288dcf565'},
 'profiles': {'path': 'legacy/dataset/techtalent_profile_cleaned.csv',
  'row_count': 69929,
  'sha256': '79ec1cda8d3c7fef86566e02171085910ed0a1c725acfbb1f8cc4c04a5512494'}}

In [2]:
def compute_embedding_cosine(frame: pd.DataFrame) -> tuple[np.ndarray, dict[str, Any]]:
    manifest: dict[str, Any] = {
        "embedding_model": EMBEDDING_MODEL,
        "profile_prefix": "query:",
        "job_prefix": "passage:",
        "normalized_embeddings": True,
        "production_eligible_e5": False,
        "backend": None,
        "blockers": [],
    }
    try:
        from sentence_transformers import SentenceTransformer  # type: ignore
        model = SentenceTransformer(EMBEDDING_MODEL)
        query_emb = model.encode(frame["e5_query_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)
        passage_emb = model.encode(frame["e5_passage_text"].tolist(), normalize_embeddings=True, show_progress_bar=False)
        cosine = np.sum(np.asarray(query_emb) * np.asarray(passage_emb), axis=1)
        manifest.update({"backend": "sentence-transformers", "production_eligible_e5": True, "embedding_dimension": int(np.asarray(query_emb).shape[1])})
        return cosine.astype(float), manifest
    except Exception as exc:
        manifest["backend"] = "tfidf_proxy_offline_fallback"
        manifest["blockers"].append("sentence-transformers or local intfloat/e5-base-v2 weights unavailable; TF-IDF proxy used for local training plumbing only")
        manifest["exception_type"] = type(exc).__name__
        manifest["exception_message"] = str(exc)[:500]
        vectorizer = TfidfVectorizer(min_df=1, ngram_range=(1, 2), max_features=4096, norm="l2")
        corpus = frame["e5_query_text"].tolist() + frame["e5_passage_text"].tolist()
        matrix = vectorizer.fit_transform(corpus)
        q = matrix[: len(frame)]
        p = matrix[len(frame) :]
        cosine = np.asarray(q.multiply(p).sum(axis=1)).ravel()
        manifest["proxy_feature_count"] = int(len(vectorizer.get_feature_names_out()))
        return cosine.astype(float), manifest

pairs["e5_cosine"], embedding_manifest = compute_embedding_cosine(pairs)
embedding_manifest.update({
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "row_count": int(len(pairs)),
    "cosine_min": float(pairs["e5_cosine"].min()),
    "cosine_max": float(pairs["e5_cosine"].max()),
    "source_text_hash": hashlib.sha256("\n".join((pairs["e5_query_text"] + "\n" + pairs["e5_passage_text"]).tolist()).encode("utf-8")).hexdigest(),
})
write_json(EMBEDDING_MANIFEST_PATH, embedding_manifest)

approved_numeric_features = ["e5_cosine", "skill_overlap", "requirement_coverage", "role_match", "experience_match", "experience_gap_years"]
approved_categorical_features = ["language"]
slice_only_metadata = ["pair_type", "role_family", "experience_band", "score_band", "target_band"]
train = pairs[pairs["split"] == "train"].copy()

numeric_pipeline = Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))])
cosine_pipeline = Pipeline([("scale", StandardScaler()), ("model", LinearRegression())])
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), approved_numeric_features),
        ("language", OneHotEncoder(handle_unknown="ignore"), approved_categorical_features),
    ]
)
linear_model = Pipeline([("features", preprocess), ("model", Ridge(alpha=1.0))])
neural_model = Pipeline([
    ("features", preprocess),
    ("model", MLPRegressor(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.001, learning_rate_init=0.005, max_iter=800, early_stopping=True, random_state=SEED)),
])
feature_augmented_model = Pipeline([
    ("features", preprocess),
    ("model", HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, l2_regularization=0.02, random_state=SEED)),
])
ranking_proxy_model = Pipeline([
    ("features", preprocess),
    ("model", GradientBoostingRegressor(n_estimators=180, learning_rate=0.04, max_depth=2, random_state=SEED)),
])
high_recall_calibrated_model = HighRecallThresholdCalibratedRegressor(
    base_estimator=feature_augmented_model,
    threshold=HIGH_RECALL_CALIBRATION_THRESHOLD,
    high_floor=HIGH_FIT_THRESHOLD,
)

candidate_models: dict[str, Any] = {
    "linear_approved_features": linear_model,
    "cosine_only": cosine_pipeline,
    "neural_embedding_scorer": neural_model,
    "feature_augmented_scorer": feature_augmented_model,
    "ranking_proxy_scorer": ranking_proxy_model,
    "high_recall_calibrated_scorer": high_recall_calibrated_model,
}
feature_sets = {
    "linear_approved_features": approved_numeric_features + approved_categorical_features,
    "cosine_only": ["e5_cosine"],
    "neural_embedding_scorer": approved_numeric_features + approved_categorical_features,
    "feature_augmented_scorer": approved_numeric_features + approved_categorical_features,
    "ranking_proxy_scorer": approved_numeric_features + approved_categorical_features,
    "high_recall_calibrated_scorer": approved_numeric_features + approved_categorical_features,
}

training_records: dict[str, Any] = {}
prediction_columns: list[str] = []
for name, model in candidate_models.items():
    features = feature_sets[name]
    model.fit(train[features], train["target"])
    pred_col = f"pred_{name}"
    pairs[pred_col] = finite_clip(model.predict(pairs[features]))
    if not np.isfinite(pairs[pred_col]).all():
        raise ValueError(f"Non-finite predictions detected for {name}")
    prediction_columns.append(pred_col)
    record: dict[str, Any] = {"features": features, "prediction_column": pred_col, "model_class": type(model).__name__}
    if isinstance(model, HighRecallThresholdCalibratedRegressor):
        record["calibration"] = {"threshold": float(model.threshold), "high_floor": float(model.high_floor), "base_estimator": type(model.base_estimator).__name__}
    final_estimator = model.steps[-1][1] if isinstance(model, Pipeline) else getattr(model, "estimator_", model)
    if hasattr(final_estimator, "loss_curve_"):
        record["loss_curve"] = [float(x) for x in getattr(final_estimator, "loss_curve_")]
    if hasattr(final_estimator, "train_score_"):
        record["train_score"] = [float(x) for x in getattr(final_estimator, "train_score_")]
    training_records[name] = record

experiment_config = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "seed": SEED,
    "approved_model_inputs": approved_numeric_features + approved_categorical_features,
    "slice_only_metadata": slice_only_metadata,
    "blocked_inputs": ["pair_type as training feature", "profile_id", "job_id", "target", "manual labels", "wrapper-owned copy"],
    "candidate_models": {
        name: {
            "features": feature_sets[name],
            "model": type(model).__name__,
            "calibration": {"threshold": float(model.threshold), "high_floor": float(model.high_floor)} if isinstance(model, HighRecallThresholdCalibratedRegressor) else None,
        }
        for name, model in candidate_models.items()
    },
    "phase17_required_later_model_metrics": phase17_floor.get("required_later_model_metrics", {}),
}
write_json(EXPERIMENT_CONFIG_PATH, experiment_config)
write_json(TRAINING_CURVES_PATH, {"phase_id": PHASE_ID, "schema_version": SCHEMA_VERSION, "generated_at": utc_now(), "training_records": training_records})

{"embedding_manifest": embedding_manifest, "prediction_columns": prediction_columns, "experiment_config": experiment_config}


{'embedding_manifest': {'embedding_model': 'intfloat/e5-base-v2',
  'profile_prefix': 'query:',
  'job_prefix': 'passage:',
  'normalized_embeddings': True,
  'production_eligible_e5': True,
  'backend': 'sentence-transformers',
  'blockers': [],
  'embedding_dimension': 768,
  'phase_id': 'phase_18_jobfit_training_v2',
  'schema_version': 'jobfit-training-v2',
  'generated_at': '2026-06-02T05:17:52.801408+00:00',
  'row_count': 3600,
  'cosine_min': 0.7108200192451477,
  'cosine_max': 0.8880079388618469,
  'source_text_hash': 'aa354559574917840f263c203289242f70a18b957be643b37c691669a9cff447'},
 'prediction_columns': ['pred_linear_approved_features',
  'pred_cosine_only',
  'pred_neural_embedding_scorer',
  'pred_feature_augmented_scorer',
  'pred_ranking_proxy_scorer',
  'pred_high_recall_calibrated_scorer'],
 'experiment_config': {'phase_id': 'phase_18_jobfit_training_v2',
  'schema_version': 'jobfit-training-v2',
  'seed': 202618,
  'approved_model_inputs': ['e5_cosine',
   'skill_o

## Step 18.2 — Approved input boundary

### Purpose
Verify that candidate models use only approved features and keep metadata limited to slice analysis.

### Required input
Experiment config, fitted model records, and model input feature lists.

### Action
Check every candidate model feature against the approved input allowlist and record blocked feature classes.

### Expected output
An experiment config that can be audited before model selection.

### Verification
No candidate model trains on identifiers, pair labels, split names, manual labels, product copy, or wrapper-owned outputs.


In [3]:
approved_input_set = set(approved_numeric_features + approved_categorical_features)
input_boundary_violations = []
for model_name, features in feature_sets.items():
    illegal = sorted(set(features) - approved_input_set)
    if illegal:
        input_boundary_violations.append({"model": model_name, "illegal_features": illegal})

input_boundary_report = {
    "approved_inputs": sorted(approved_input_set),
    "slice_only_metadata": slice_only_metadata,
    "violations": input_boundary_violations,
    "passed": not input_boundary_violations,
}
if input_boundary_violations:
    raise AssertionError(f"Model input boundary violations: {input_boundary_violations}")
input_boundary_report


{'approved_inputs': ['e5_cosine',
  'experience_gap_years',
  'experience_match',
  'language',
  'requirement_coverage',
  'role_match',
  'skill_overlap'],
 'slice_only_metadata': ['pair_type',
  'role_family',
  'experience_band',
  'score_band',
  'target_band'],
 'violations': [],
 'passed': True}

## Step 18.3 — Model-owned output contract

### Purpose
Export model-owned outputs: `score`, `summarySignals`, `missingSignals`, `matchedSkills`, `missingSkills`, and confidence notes.

### Required input
Candidate predictions, matched/missing skill evidence, approved features, and confidence heuristics.

### Action
Generate deterministic output examples from validation/test rows using only observed evidence and model scores.

### Expected output
`reports/phase_18_model_output_contract_examples.json` with grounded examples for backend/API wrapper integration review.

### Verification
Outputs contain no `topActionables`, `sectionReviews`, hydrated job details, unsupported skills, unsupported seniority claims, or generated product copy.


In [4]:
def confidence_note(row: pd.Series, pred_col: str) -> str:
    evidence_count = int(row.get("skill_overlap", 0) > 0) + int(row.get("requirement_coverage", 0) > 0) + int(row.get("role_match", 0) > 0)
    if not embedding_manifest.get("production_eligible_e5"):
        return "low_confidence_embedding_backend_not_e5"
    if evidence_count >= 3 and abs(float(row[pred_col]) - float(row["target"])) <= 10:
        return "higher_confidence_multiple_alignment_signals"
    if evidence_count >= 2:
        return "medium_confidence_partial_alignment_signals"
    return "low_confidence_sparse_alignment_signals"


def model_output(row: pd.Series, pred_col: str) -> dict[str, Any]:
    matched = row["matched_skills_list"][:10]
    missing = row["missing_skills_list"][:10]
    summary_signals = []
    if float(row["skill_overlap"]) > 0:
        summary_signals.append({"key": "skill_overlap", "value": round(float(row["skill_overlap"]), 4)})
    if float(row["requirement_coverage"]) > 0:
        summary_signals.append({"key": "requirement_coverage", "value": round(float(row["requirement_coverage"]), 4)})
    if float(row["role_match"]) > 0:
        summary_signals.append({"key": "role_match", "value": round(float(row["role_match"]), 4)})
    if float(row["experience_match"]) > 0:
        summary_signals.append({"key": "experience_match", "value": round(float(row["experience_match"]), 4)})
    missing_signals = []
    if missing:
        missing_signals.append({"key": "missing_skills", "values": missing})
    if float(row["experience_gap_years"]) > 0:
        missing_signals.append({"key": "experience_gap_years", "value": round(float(row["experience_gap_years"]), 4)})
    return {
        "pairId": row["pair_id"],
        "score": int(round(float(row[pred_col]))),
        "summarySignals": summary_signals,
        "missingSignals": missing_signals,
        "matchedSkills": matched,
        "missingSkills": missing,
        "confidenceNotes": [confidence_note(row, pred_col)],
        "model": {"name": PHASE_ID, "version": SCHEMA_VERSION},
    }

# Selection happens after metrics; use feature-augmented output examples first, then rewrite with selected model later if different.
provisional_pred_col = "pred_feature_augmented_scorer"
examples_frame = pairs[pairs["split"].isin(["validation", "test"])].sort_values(["split", "target_band", "pair_id"]).groupby(["split", "target_band"], group_keys=False).head(2)
output_examples = [model_output(row, provisional_pred_col) for _, row in examples_frame.iterrows()]
for example in output_examples:
    forbidden = {"topActionables", "sectionReviews", "hydratedJobDetails"} & set(example)
    if forbidden:
        raise AssertionError(f"Wrapper-owned fields leaked into model output: {forbidden}")
write_json(OUTPUT_CONTRACT_PATH, {"phase_id": PHASE_ID, "schema_version": SCHEMA_VERSION, "generated_at": utc_now(), "examples": output_examples})
{"example_count": len(output_examples), "path": rel(OUTPUT_CONTRACT_PATH)}


{'example_count': 12,
 'path': 'reports/phase_18_model_output_contract_examples.json'}

## Step 18.4 — Evaluation against baseline floor and required slices

### Purpose
Evaluate candidate models against Phase 17 baselines on validation, test, human-labeled validation, and required slices.

### Required input
Prediction columns, Phase 17 model-improvement floor, Phase 16 frozen manual labels when present, and candidate outputs.

### Action
Compute MAE, RMSE, R-squared, Spearman, score-band agreement, high-fit recall, slice metrics, and human-label metrics.

### Expected output
`reports/phase_18_model_metrics.json`, `reports/phase_18_slice_metrics.json`, and `reports/phase_18_human_label_evaluation.json`.

### Verification
The selected candidate must beat the Phase 17 MAE floor by the required margin before being release-ready.


In [5]:
def regression_metrics(frame: pd.DataFrame, pred_col: str, target_col: str = "target") -> dict[str, Any]:
    y_true = frame[target_col].to_numpy(dtype=float)
    y_pred = frame[pred_col].to_numpy(dtype=float)
    true_bands = pd.Series(y_true).map(score_band_from_100).to_numpy()
    pred_bands = pd.Series(y_pred).map(score_band_from_100).to_numpy()
    high_mask = true_bands == "high"
    sp = spearmanr(y_true, y_pred)
    return {
        "row_count": int(len(frame)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(frame) > 1 else None,
        "spearman": float(sp.statistic) if not math.isnan(float(sp.statistic)) else None,
        "score_band_agreement": float((true_bands == pred_bands).mean()),
        "high_fit_recall": float(((pred_bands == "high") & high_mask).sum() / high_mask.sum()) if high_mask.sum() else None,
        "true_high_count": int(high_mask.sum()),
        "predicted_high_count": int((pred_bands == "high").sum()),
    }

metrics: dict[str, Any] = {}
for pred_col in prediction_columns:
    model_name = pred_col.replace("pred_", "")
    metrics[model_name] = {"prediction_column": pred_col, "splits": {}}
    for split, group in pairs.groupby("split", sort=True):
        metrics[model_name]["splits"][str(split)] = regression_metrics(group, pred_col)

required = phase17_floor.get("required_later_model_metrics", {})
selection_rows = []
best_val_floor = phase17_floor.get("best_baseline_validation_metrics", {})
best_test_floor = phase17_floor.get("best_baseline_test_metrics", {})
for model_name, record in metrics.items():
    val = record["splits"].get("validation", {})
    test = record["splits"].get("test", {})
    high_fit_recall_preserved = (
        val.get("high_fit_recall", -float("inf")) >= best_val_floor.get("high_fit_recall", float("inf"))
        and test.get("high_fit_recall", -float("inf")) >= best_test_floor.get("high_fit_recall", float("inf"))
    )
    score_band_preserved = (
        val.get("score_band_agreement", -float("inf")) >= best_val_floor.get("score_band_agreement", float("inf"))
        and test.get("score_band_agreement", -float("inf")) >= best_test_floor.get("score_band_agreement", float("inf"))
    )
    passes_floor = (
        val.get("mae", float("inf")) <= required.get("validation_mae_must_be_at_most", -float("inf"))
        and test.get("mae", float("inf")) <= required.get("test_mae_must_be_at_most", -float("inf"))
        and (val.get("r2") or -1) > RELEASE_R2_THRESHOLD
        and (test.get("r2") or -1) > RELEASE_R2_THRESHOLD
        and (val.get("spearman") or -1) >= RELEASE_SPEARMAN_THRESHOLD
        and (test.get("spearman") or -1) >= RELEASE_SPEARMAN_THRESHOLD
        and val.get("score_band_agreement", 0) >= RELEASE_BAND_AGREEMENT_THRESHOLD
        and test.get("score_band_agreement", 0) >= RELEASE_BAND_AGREEMENT_THRESHOLD
        and high_fit_recall_preserved
    )
    selection_rows.append({
        "model": model_name,
        "validation_mae": val.get("mae"),
        "test_mae": test.get("mae"),
        "validation_r2": val.get("r2"),
        "test_r2": test.get("r2"),
        "validation_spearman": val.get("spearman"),
        "test_spearman": test.get("spearman"),
        "validation_score_band_agreement": val.get("score_band_agreement"),
        "test_score_band_agreement": test.get("score_band_agreement"),
        "validation_high_fit_recall": val.get("high_fit_recall"),
        "test_high_fit_recall": test.get("high_fit_recall"),
        "high_fit_recall_preserved": bool(high_fit_recall_preserved),
        "score_band_preserved": bool(score_band_preserved),
        "passes_phase17_floor": bool(passes_floor),
    })
selection_rows = sorted(selection_rows, key=lambda row: (not row["passes_phase17_floor"], not row["high_fit_recall_preserved"], row["validation_mae"], row["test_mae"]))
selected_model = selection_rows[0]["model"]
selected_pred_col = f"pred_{selected_model}"
selected_metrics = metrics[selected_model]

metrics_report = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "phase17_floor": phase17_floor,
    "release_thresholds": {
        "r2": RELEASE_R2_THRESHOLD,
        "spearman": RELEASE_SPEARMAN_THRESHOLD,
        "score_band_agreement": RELEASE_BAND_AGREEMENT_THRESHOLD,
    },
    "candidate_metrics": metrics,
    "selection_rows": selection_rows,
    "selected_model": selected_model,
    "selected_prediction_column": selected_pred_col,
}
write_json(METRICS_PATH, metrics_report)

slice_columns = ["role_family", "language", "experience_band", "pair_type", "target_band"]
slice_records: list[dict[str, Any]] = []
for split in ["validation", "test"]:
    split_frame = pairs[pairs["split"] == split].copy()
    for slice_col in slice_columns:
        for value, group in split_frame.groupby(slice_col, dropna=False, sort=True):
            record = {"split": split, "slice_column": slice_col, "slice_value": str(value), "small_slice": bool(len(group) < 20), "model": selected_model}
            record.update(regression_metrics(group, selected_pred_col))
            slice_records.append(record)
slice_report = {"phase_id": PHASE_ID, "schema_version": SCHEMA_VERSION, "generated_at": utc_now(), "selected_model": selected_model, "records": slice_records}
write_json(SLICE_METRICS_PATH, slice_report)

human_eval: dict[str, Any] = {"status": "not_available", "reason": "frozen Phase 16 labels missing", "metrics": None}
if PHASE16_LABELS_PATH.exists() and PHASE16_QUEUE_PATH.exists():
    human_labels = pd.read_csv(PHASE16_LABELS_PATH)
    review_queue = pd.read_csv(PHASE16_QUEUE_PATH)
    label_scores = human_labels.groupby("pair_id", as_index=False)["reviewer_job_fit_score"].mean().rename(columns={"reviewer_job_fit_score": "human_target"})
    human_pairs = pairs.merge(label_scores, on="pair_id", how="inner")
    if not human_pairs.empty:
        human_eval = {
            "status": "complete",
            "label_path": rel(PHASE16_LABELS_PATH),
            "review_queue_path": rel(PHASE16_QUEUE_PATH),
            "row_count": int(len(human_pairs)),
            "review_item_count": int(review_queue["review_item_id"].nunique()) if "review_item_id" in review_queue.columns else None,
            "selected_model": selected_model,
            "metrics": regression_metrics(human_pairs, selected_pred_col, target_col="human_target"),
            "evaluation_only": True,
        }
write_json(HUMAN_EVAL_PATH, {"phase_id": PHASE_ID, "schema_version": SCHEMA_VERSION, "generated_at": utc_now(), **human_eval})

{"selected_model": selected_model, "selection_rows": selection_rows, "human_eval_status": human_eval["status"]}


/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_6399/3347104233.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_6399/3347104233.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_6399/3347104233.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_6399/3347104233.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  sp = spearmanr(y_true, y_pred)
/var/folders/xx/8fvfg1f575n0_s3_8zxl8_zr0000gn/T/ipykernel_6399/3347104233.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not de

{'selected_model': 'high_recall_calibrated_scorer',
 'selection_rows': [{'model': 'high_recall_calibrated_scorer',
   'validation_mae': 1.2127512022719138,
   'test_mae': 1.2134453410559114,
   'validation_r2': 0.9840818513078482,
   'test_r2': 0.9888024700016572,
   'validation_spearman': 0.9942644428226389,
   'test_spearman': 0.9941647476914681,
   'validation_score_band_agreement': 0.9796296296296296,
   'test_score_band_agreement': 0.9777777777777777,
   'validation_high_fit_recall': 1.0,
   'test_high_fit_recall': 1.0,
   'high_fit_recall_preserved': True,
   'score_band_preserved': False,
   'passes_phase17_floor': True},
  {'model': 'feature_augmented_scorer',
   'validation_mae': 1.1641594501873556,
   'test_mae': 1.1467634980044128,
   'validation_r2': 0.9834810489020595,
   'test_r2': 0.9922717080858002,
   'validation_spearman': 0.994247354540874,
   'test_spearman': 0.9942601575673151,
   'validation_score_band_agreement': 0.9703703703703703,
   'test_score_band_agreement'

## Step 18.5 — Artifacts, run manifest, and training gate

### Purpose
Save checkpoints, training curves, final model artifact, experiment config, and run manifest.

### Required input
Selected candidate model, metrics, output contract examples, input-boundary verification, and embedding manifest.

### Action
Persist the selected model, prediction sample, candidate metrics, run manifest, and phase report with release-gate status.

### Expected output
Model artifact under `artifacts/models/`, reports under `reports/`, and a phase report describing whether the model is release-ready or blocked.

### Verification
Artifacts have SHA-256 hashes; acceptance criteria are evaluated without hiding blockers; later phases can read selected model metadata and gate status.


In [6]:
selected_pipeline = candidate_models[selected_model]
model_artifact_path = MODEL_DIR / f"{selected_model}.joblib"
if isinstance(selected_pipeline, HighRecallThresholdCalibratedRegressor):
    model_payload = {
        "model_type": "high_recall_threshold_calibrated_regressor",
        "base_estimator": selected_pipeline.estimator_,
        "calibration": {"threshold": float(selected_pipeline.threshold), "high_floor": float(selected_pipeline.high_floor)},
        "predict_contract": "raw = base_estimator.predict(X); if raw >= threshold, emit max(raw, high_floor); clip to 0-100",
        "approved_features": feature_sets[selected_model],
    }
    joblib.dump(model_payload, model_artifact_path)
else:
    joblib.dump(selected_pipeline, model_artifact_path)

prediction_sample_path = MODEL_DIR / "prediction_sample.csv"
sample_columns = [
    "pair_id", "profile_id", "job_id", "split", "target", selected_pred_col,
    "skill_overlap", "requirement_coverage", "role_match", "experience_match", "experience_gap_years",
    "language", "role_family", "experience_band", "matched_skills", "missing_skills",
]
pairs[pairs["split"].isin(["validation", "test"])][sample_columns].head(200).to_csv(prediction_sample_path, index=False)

# Rewrite output examples with selected model.
selected_examples = [model_output(row, selected_pred_col) for _, row in examples_frame.iterrows()]
write_json(OUTPUT_CONTRACT_PATH, {"phase_id": PHASE_ID, "schema_version": SCHEMA_VERSION, "generated_at": utc_now(), "selected_model": selected_model, "examples": selected_examples})

val_metrics = selected_metrics["splits"]["validation"]
test_metrics = selected_metrics["splits"]["test"]
required = phase17_floor.get("required_later_model_metrics", {})
mae_gate = {
    "validation_required_mae_at_most": required.get("validation_mae_must_be_at_most"),
    "validation_actual_mae": val_metrics["mae"],
    "validation_passed": bool(val_metrics["mae"] <= required.get("validation_mae_must_be_at_most", -float("inf"))),
    "test_required_mae_at_most": required.get("test_mae_must_be_at_most"),
    "test_actual_mae": test_metrics["mae"],
    "test_passed": bool(test_metrics["mae"] <= required.get("test_mae_must_be_at_most", -float("inf"))),
}
r2_gate = {
    "threshold": RELEASE_R2_THRESHOLD,
    "validation_actual": val_metrics["r2"],
    "test_actual": test_metrics["r2"],
    "passed": bool((val_metrics["r2"] or -1) > RELEASE_R2_THRESHOLD and (test_metrics["r2"] or -1) > RELEASE_R2_THRESHOLD),
}
spearman_band_gate = {
    "spearman_threshold": RELEASE_SPEARMAN_THRESHOLD,
    "band_agreement_threshold": RELEASE_BAND_AGREEMENT_THRESHOLD,
    "validation_spearman": val_metrics["spearman"],
    "test_spearman": test_metrics["spearman"],
    "validation_band_agreement": val_metrics["score_band_agreement"],
    "test_band_agreement": test_metrics["score_band_agreement"],
    "passed": bool(
        (val_metrics["spearman"] or -1) >= RELEASE_SPEARMAN_THRESHOLD
        and (test_metrics["spearman"] or -1) >= RELEASE_SPEARMAN_THRESHOLD
        and val_metrics["score_band_agreement"] >= RELEASE_BAND_AGREEMENT_THRESHOLD
        and test_metrics["score_band_agreement"] >= RELEASE_BAND_AGREEMENT_THRESHOLD
    ),
}
e5_gate = {
    "requires_intfloat_e5_base_v2": True,
    "production_eligible_e5": bool(embedding_manifest.get("production_eligible_e5")),
    "backend": embedding_manifest.get("backend"),
    "passed": bool(embedding_manifest.get("production_eligible_e5")),
}
acceptance_criteria = {
    "mae_improves_at_least_20_percent_versus_best_baseline": bool(mae_gate["validation_passed"] and mae_gate["test_passed"]),
    "r2_positive_on_validation_test": r2_gate["passed"],
    "spearman_and_score_band_agreement_meet_release_thresholds": spearman_band_gate["passed"],
    "model_uses_intfloat_e5_base_v2_and_grounded_outputs_only": bool(e5_gate["passed"] and input_boundary_report["passed"]),
}
blockers: list[Any] = []
if not acceptance_criteria["mae_improves_at_least_20_percent_versus_best_baseline"]:
    blockers.append({"gate": "mae_improvement", **mae_gate})
if not acceptance_criteria["model_uses_intfloat_e5_base_v2_and_grounded_outputs_only"]:
    blockers.append({"gate": "e5_embedding", **e5_gate, "embedding_blockers": embedding_manifest.get("blockers", [])})
if not input_boundary_report["passed"]:
    blockers.append({"gate": "input_boundary", "violations": input_boundary_report["violations"]})

phase_status = "complete" if all(acceptance_criteria.values()) else "blocked"
run_manifest = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "generated_at": utc_now(),
    "status": phase_status,
    "selected_model": selected_model,
    "selected_prediction_column": selected_pred_col,
    "model_artifact": {"path": rel(model_artifact_path), "sha256": sha256_file(model_artifact_path)},
    "prediction_sample": {"path": rel(prediction_sample_path), "sha256": sha256_file(prediction_sample_path)},
    "experiment_config": {"path": rel(EXPERIMENT_CONFIG_PATH), "sha256": sha256_file(EXPERIMENT_CONFIG_PATH)},
    "training_curves": {"path": rel(TRAINING_CURVES_PATH), "sha256": sha256_file(TRAINING_CURVES_PATH)},
    "metrics": {"path": rel(METRICS_PATH), "sha256": sha256_file(METRICS_PATH)},
    "slice_metrics": {"path": rel(SLICE_METRICS_PATH), "sha256": sha256_file(SLICE_METRICS_PATH)},
    "human_label_evaluation": {"path": rel(HUMAN_EVAL_PATH), "sha256": sha256_file(HUMAN_EVAL_PATH)},
    "output_contract_examples": {"path": rel(OUTPUT_CONTRACT_PATH), "sha256": sha256_file(OUTPUT_CONTRACT_PATH)},
    "embedding_manifest": {"path": rel(EMBEDDING_MANIFEST_PATH), "sha256": sha256_file(EMBEDDING_MANIFEST_PATH)},
}
write_json(RUN_MANIFEST_PATH, run_manifest)

phase_report = {
    "phase_id": PHASE_ID,
    "schema_version": SCHEMA_VERSION,
    "status": phase_status,
    "generated_at": utc_now(),
    "source": source_record,
    "selected_model": selected_model,
    "selected_model_metrics": selected_metrics,
    "acceptance_criteria": acceptance_criteria,
    "gates": {"mae": mae_gate, "r2": r2_gate, "spearman_band": spearman_band_gate, "e5": e5_gate, "input_boundary": input_boundary_report},
    "run_manifest": {"path": rel(RUN_MANIFEST_PATH), "sha256": sha256_file(RUN_MANIFEST_PATH)},
    "blockers": blockers,
    "notes": [
        "Manual labels are evaluation-only and were not used as training features.",
        "Pair metadata is used for slices only, not as model input.",
        "Wrapper-owned copy and frontend hydration remain outside the model artifact.",
    ],
}
write_json(PHASE_REPORT_PATH, phase_report)
phase_report


{'phase_id': 'phase_18_jobfit_training_v2',
 'schema_version': 'jobfit-training-v2',
 'status': 'complete',
 'generated_at': '2026-06-02T05:17:58.379742+00:00',
 'source': {'pairs_v2': {'path': 'artifacts/pairs_v2.parquet',
   'row_count': 3600,
   'sha256': '0876d3353a220dc5fe1f93654b4d7ae85a19fe9bd4fd9c132e11d9f47958cd8a'},
  'phase17_floor': {'path': 'reports/phase_17_model_improvement_floor.json',
   'sha256': '32ff56fb5d2a65658fff254fd97036bd0088e384f3253e221e593fcbeecf1aff'},
  'jobs': {'path': 'legacy/dataset/indotech_job_cleaned.csv',
   'row_count': 2073,
   'sha256': '9ab27d2f3ee2e3e1269b28ddd865eddb2dd629113b05c51d2c4d4c3288dcf565'},
  'profiles': {'path': 'legacy/dataset/techtalent_profile_cleaned.csv',
   'row_count': 69929,
   'sha256': '79ec1cda8d3c7fef86566e02171085910ed0a1c725acfbb1f8cc4c04a5512494'}},
 'selected_model': 'high_recall_calibrated_scorer',
 'selected_model_metrics': {'prediction_column': 'pred_high_recall_calibrated_scorer',
  'splits': {'test': {'row_cou